# Train YOLOv8 on aerial parking-lot vehicles

Fine-tunes YOLOv8n on **CARPK** — a dataset of drone/rooftop photos looking straight down on parking lots, with every car boxed (Hsieh et al., ICCV 2017). This is the failure mode the Day32 project hits with the stock COCO-pretrained `yolov8n.pt`: from directly overhead, cars are small, foreshortened, and don't look like the mostly side-on/three-quarter view COCO was trained on, so recall on `aerial_*` sample videos is poor. Training on CARPK teaches the model what a car looks like from that angle.

**What this notebook does:**
1. Downloads CARPK (YOLOv8 format) from Roboflow Universe
2. Fine-tunes `yolov8n.pt` on it, at `imgsz=1280` to match the local project's inference resolution
3. Validates and visualizes predictions
4. Runs the trained model frame-by-frame over an uploaded video (optional)
5. Downloads `best.pt` so you can drop it into the local `parking_detection.py` pipeline

**Before you start:** `Runtime -> Change runtime type -> T4 GPU`, and get a free Roboflow API key from [app.roboflow.com/settings/api](https://app.roboflow.com/settings/api) (needed to pull the dataset).

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q ultralytics roboflow

## 1. Download the dataset

[CARPK on Roboflow Universe](https://universe.roboflow.com/elpida-eleftheriadi/carpk-xk8e1/dataset/1) — 1,567 aerial parking-lot images, single class `car`. Paste your API key below (Settings -> API Keys -> Private API Key on roboflow.com — the key itself is private, don't commit it anywhere).

In [ ]:
ROBOFLOW_API_KEY = "YOUR_API_KEY_HERE"

from roboflow import Roboflow

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("elpida-eleftheriadi").project("carpk-xk8e1")
dataset = project.version(1).download("yolov8")
DATA_YAML = f"{dataset.location}/data.yaml"
print(DATA_YAML)

In [ ]:
import yaml
from pathlib import Path

DATASET_ROOT = Path(dataset.location)
cfg = yaml.safe_load((DATASET_ROOT / "data.yaml").read_text())
print("classes:", cfg["names"])

# Roboflow's YOLOv8 export always uses this on-disk layout (train/images,
# valid/images, test/images) - trust that, not the relative paths written
# inside data.yaml (those get resolved by Ultralytics' own training code,
# not by naively joining them onto DATASET_ROOT here).
SPLIT_DIRS = {"train": "train", "val": "valid", "test": "test"}

def split_image_dir(split):
    d = DATASET_ROOT / SPLIT_DIRS[split] / "images"
    return d if d.is_dir() else None

for split in ("train", "val", "test"):
    d = split_image_dir(split)
    n = (len(list(d.glob("*.jpg"))) + len(list(d.glob("*.png")))) if d else 0
    print(f"{split}: {n} images ({d})")

Quick sanity check before spending compute on training — one training image with its ground-truth boxes drawn on top.

In [ ]:
import cv2
import matplotlib.pyplot as plt

train_img_dir = split_image_dir("train")
sample_img = sorted(train_img_dir.glob("*.jpg"))[0]
sample_lbl = Path(str(sample_img).replace("/images/", "/labels/")).with_suffix(".txt")

img = cv2.cvtColor(cv2.imread(str(sample_img)), cv2.COLOR_BGR2RGB)
h, w = img.shape[:2]
for line in sample_lbl.read_text().splitlines():
    cls, cx, cy, bw, bh = map(float, line.split())
    x1, y1 = int((cx - bw / 2) * w), int((cy - bh / 2) * h)
    x2, y2 = int((cx + bw / 2) * w), int((cy + bh / 2) * h)
    cv2.rectangle(img, (x1, y1), (x2, y2), (255, 0, 0), 2)

plt.figure(figsize=(10, 10))
plt.imshow(img)
plt.axis("off")
plt.title(sample_img.name)
plt.show()

## 2. Fine-tune YOLOv8n

Starts from the same `yolov8n.pt` COCO checkpoint the local project ships, so the result is a drop-in replacement, not a different architecture. `imgsz=1280` matches `DEFAULT_IMGSZ` in `parking_detection.py` — training and inference resolution should line up, especially for small aerial objects. `batch=-1` lets Ultralytics auto-size the batch to fit whatever GPU Colab hands you.

~60 epochs on CARPK at this resolution takes roughly 1-1.5h on a free T4. Early stopping (`patience=15`) will cut it short if validation mAP stops improving.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
results = model.train(
    data=DATA_YAML,
    epochs=60,
    imgsz=1280,
    batch=-1,
    patience=15,
    device=0,
    project="aerial_vehicle_yolo",
    name="carpk_yolov8n",
)

## 3. Validate

In [ ]:
best_weights = f"{results.save_dir}/weights/best.pt"
trained = YOLO(best_weights)
metrics = trained.val(data=DATA_YAML, imgsz=1280)
print(f"mAP50:    {metrics.box.map50:.3f}")
print(f"mAP50-95: {metrics.box.map:.3f}")
print(f"precision:{metrics.box.mp:.3f}")
print(f"recall:   {metrics.box.mr:.3f}")

In [ ]:
import random

val_img_dir = split_image_dir("val")
val_imgs = random.sample(sorted(val_img_dir.glob("*.jpg")), k=4)
preds = trained.predict(val_imgs, imgsz=1280, conf=0.25, verbose=False)

fig, axes = plt.subplots(2, 2, figsize=(14, 14))
for ax, pred in zip(axes.ravel(), preds):
    ax.imshow(cv2.cvtColor(pred.plot(), cv2.COLOR_BGR2RGB))
    ax.axis("off")
    ax.set_title(f"{len(pred.boxes)} cars found")
plt.tight_layout()
plt.show()

## 4. Run it over a real video, frame by frame

Upload one of the project's `aerial_*.mp4` sample clips (or any aerial parking-lot footage) to see the fine-tuned model detect vehicles across the whole clip, not just single frames. Skip this cell and the next if you just want the weights.

In [ ]:
from google.colab import files

uploaded = files.upload()
video_path = next(iter(uploaded))
print("using:", video_path)

In [ ]:
cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS) or 25
fw, fh = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

out_path = "detected_" + Path(video_path).name
writer = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (fw, fh))

frame_count = 0
while True:
    ok, frame = cap.read()
    if not ok:
        break
    result = trained.predict(frame, imgsz=1280, conf=0.25, verbose=False)[0]
    writer.write(result.plot())
    frame_count += 1

cap.release()
writer.release()
print(f"processed {frame_count} frames -> {out_path}")

In [ ]:
files.download(out_path)

## 5. Download the trained weights

In [ ]:
files.download(best_weights)

## 6. Bring it back into the local project

Drop the downloaded weights file next to `parking_detection.py` (e.g. rename it `vehicle_detection_aerial.pt`), then:

```bash
python parking_detection.py sample_videos/aerial_mall_lot.mp4 --layout sample_videos/aerial_mall_lot_spaces.json --model vehicle_detection_aerial.pt
```

No code changes needed — `parking_detection.py` now derives its vehicle-class filter from whatever model you pass it (matching class names like `car`, `bus`, `truck` against that model's own label list) instead of a hardcoded set of COCO ids, so a single-class CARPK model works the same way the stock COCO model does.